In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import ParameterGrid
from sklearn.manifold import TSNE
import umap

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca": "../../data/reduced/dataPCA.csv",
    "fs": "../../data/reduced/dataFS.csv"
}

outDir = "../../results/dbscan"

In [3]:
def plotDB(X, labels, method, name):
    plt.figure(figsize=(6,5))
    noise = labels == -1

    plt.scatter(X[noise,0], X[noise,1], c="lightgray", s=5, label="Noise")
    plt.scatter(X[~noise,0], X[~noise,1], c=labels[~noise], cmap="tab10", s=5)

    plt.title(f"DBSCAN {method} ({name})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/dbscan_{method}_{name}.png", dpi=200)
    plt.close()

In [4]:
param_grid = {
    "eps": [0.2, 0.5, 0.8, 1.0, 1.5],
    "min_samples": [5, 10, 20]
}

results = []

for name, path in datasets.items():
    
    df = pd.read_csv(path)

    if "Class" in df.columns:
        y_true = df["Class"].values
        X = df.drop(columns=["Class"])
    else:
        y_true = None
        X = df.copy()

    if "V129" in X.columns:
        X = X.drop(columns=["V129"])

    X_np = X.values

    best_score = -1
    best_params = None
    best_labels = None

    for params in ParameterGrid(param_grid):
        db = DBSCAN(**params, n_jobs=-1)
        labels = db.fit_predict(X_np)

        if len(set(labels)) <= 2:
            continue

        mask = labels != -1
        sil = silhouette_score(X_np[mask], labels[mask])

        if sil > best_score:
            best_score = sil
            best_params = params
            best_labels = labels

    labels = best_labels

    num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = np.mean(labels == -1)

    mask = labels != -1
    if len(set(labels[mask])) > 1:
        sil = silhouette_score(X_np[mask], labels[mask])
        db  = davies_bouldin_score(X_np[mask], labels[mask])
    else:
        sil, db = np.nan, np.nan

    if y_true is not None:
        ari = adjusted_rand_score(y_true, labels)
        nmi = normalized_mutual_info_score(y_true, labels)
    else:
        ari, nmi = np.nan, np.nan

    out = X.copy()
    out["cluster"] = labels
    out.to_csv(f"{outDir}/dbscan_{name}_clusters.csv", index=False)

    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_np)
    plotDB(X_umap, labels, "UMAP", name)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X_np)
    plotDB(X_tsne, labels, "TSNE", name)

    results.append({
        "Dataset": name,
        "Algorithm": "DBSCAN",
        "eps": best_params["eps"],
        "min_samples": best_params["min_samples"],
        "Clusters": num_clusters,
        "NoiseRatio": noise_ratio,
        "Silhouette": sil,
        "Davies_Bouldin": db,
        "ARI": ari,
        "NMI": nmi
    })

C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
summary = pd.DataFrame(results)
summary.to_csv(f"{outDir}/dbscan_summary.csv", index=False)
print(summary)

  Dataset Algorithm  eps  min_samples  Clusters  NoiseRatio  Silhouette  \
0    full    DBSCAN  0.2           20        61    0.820273    0.692178   
1     pca    DBSCAN  0.2           20       114    0.574047    0.553047   
2      fs    DBSCAN  1.5           20         3    0.005536    0.617980   

   Davies_Bouldin       ARI       NMI  
0        0.446130  0.011587  0.224513  
1        0.446297       NaN       NaN  
2        0.367787       NaN       NaN  
